# Layer-3 Resource Probe — Denodo Data Catalog (dev)
### OAuth (Authorization Code flow) edition

**Goal:** arrive at the Maxen meeting with the sentence:
> "Of the 15,014 views on dev, **N** have layer-3 resources, distributed across these metadata fields."

**Method:**
1. Enumerate all views in `dataportal` (paginated).
2. For each view, fetch `view-details` keyed on `(name, db)` and scan for resource signals:
   - **(a)** explicit attachment/documentation fields, if the endpoint exposes them
   - **(b)** URLs embedded in RICH_TEXT property values after HTML stripping
   - **(c)** properties whose *name* suggests files/docs (from the 341-property catalog)
3. Aggregate counts per signal type; save raw hits to SQLite (`probe_results.db`).

**Auth:** reuses the `OAuthManager` from the Authorization Code flow notebook. The browser
popup appears **once** at the start; after that, token refresh happens automatically —
important because the full sweep (~40–50 min) outlives a typical 1-hour access token.

**Design notes:** resumable (checkpoints every ~500 views), sequential with a small
sleep — dev is a shared environment, be polite.

## 1. Imports & environment

Credentials (`AUTH_FLOW_CLIENT_ID`, `AUTH_FLOW_CLIENT_SECRET`, `REDIRECT_URI`,
`AUTH_URL`, `TOKEN_URL`, `SCOPE`) are set as **Windows User Environment Variables**
(via "Edit environment variables for your account") — there is no `.env` file.
`os.getenv()` reads them directly, so no `dotenv` loading is needed.
The verification cell below confirms all six are present before authenticating.

In [1]:
import html
import re
import sqlite3
import time
import os
from collections import Counter

import requests
import webview
import truststore
from requests_oauthlib import OAuth2Session

truststore.inject_into_ssl()   # handles LANL internal TLS certs

# Credentials live in Windows User Environment Variables (no .env file).
# Verify all six are present before going further:
required = ["AUTH_FLOW_CLIENT_ID", "AUTH_FLOW_CLIENT_SECRET",
            "REDIRECT_URI", "AUTH_URL", "TOKEN_URL", "SCOPE"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise EnvironmentError(f"Missing environment variables: {missing}")
print("All 6 env vars present \u2713")

All 6 env vars present ✓


## 2. OAuthManager

Same class as the Authorization Code flow notebook: initial browser-popup auth,
automatic token refresh, and `get()`/`post()` helpers that inject the Bearer header.

In [2]:
class OAuthManager:
    """
    Complete OAuth manager that handles initial auth AND refresh
    """
    def __init__(self, client_id, client_secret, redirect_uri, auth_url, token_url, scope):
        self.client_id = client_id
        self.client_secret = client_secret
        self.redirect_uri = redirect_uri
        self.auth_url = auth_url
        self.token_url = token_url
        self.scope = scope

        # These will be set after authentication
        self.oauth = None
        self.token = None
        self.access_token = None
        self.refresh_token = None
        self.token_expiry = None

    def authenticate(self):
        """
        Perform initial OAuth flow with browser popup
        Returns True if successful
        """
        self.oauth = OAuth2Session(
            self.client_id,
            redirect_uri=self.redirect_uri,
            scope=self.scope
        )

        authorization_url, state = self.oauth.authorization_url(self.auth_url)

        print('Authenticating...')

        authorization_response = self._launch_browser_auth(authorization_url)

        if not authorization_response:
            print("\u2717 Authentication failed")
            return False

        self.token = self.oauth.fetch_token(
            self.token_url,
            authorization_response=authorization_response,
            client_secret=self.client_secret
        )

        self.access_token = self.token["access_token"]
        self.refresh_token = self.token.get("refresh_token")
        self.token_expiry = time.time() + self.token.get("expires_in", 3600)

        print("\u2713 Authentication successful")
        return True

    def _launch_browser_auth(self, authorization_url):
        """Launch webview for OAuth authorization"""
        authorization_response = None

        def on_loaded():
            nonlocal authorization_response
            current_url = window.get_current_url()
            if current_url and self.redirect_uri in current_url:
                authorization_response = current_url
                print('\u2713 Captured Authorization')
                window.hide()
                time.sleep(2.5)
                window.destroy()

        window = webview.create_window(
            'OAuth Authorization',
            authorization_url,
            width=800,
            height=600,
            resizable=True,
            on_top=True
        )

        window.events.loaded += on_loaded
        webview.start()

        return authorization_response

    def _is_token_expired(self):
        """Check if access token is expired"""
        if not self.token_expiry:
            return True
        return time.time() >= (self.token_expiry - 60)

    def _refresh_access_token(self):
        """Refresh the access token"""
        if not self.refresh_token:
            print("\u26a0 No refresh token available, re-authenticating...")
            return self.authenticate()

        try:
            new_token = self.oauth.refresh_token(
                self.token_url,
                refresh_token=self.refresh_token,
                client_id=self.client_id,
                client_secret=self.client_secret
            )

            self.token = new_token
            self.access_token = new_token["access_token"]
            self.refresh_token = new_token.get("refresh_token", self.refresh_token)
            self.token_expiry = time.time() + new_token.get("expires_in", 3600)

            print("\u2713 Token refreshed successfully")
            return True
        except Exception as e:
            print(f"\u2717 Refresh failed: {e}")
            print("Re-authenticating...")
            return self.authenticate()

    def _ensure_authenticated(self):
        """Ensure we have a valid token, authenticate if needed"""
        if not self.access_token or self._is_token_expired():
            if self.refresh_token:
                self._refresh_access_token()
            else:
                self.authenticate()

    def get(self, url, **kwargs):
        """Make GET request with automatic authentication/refresh"""
        self._ensure_authenticated()
        headers = kwargs.pop('headers', {})
        headers['Authorization'] = f'Bearer {self.access_token}'
        headers['Content-Type'] = headers.get('Content-Type', 'application/json')
        return requests.get(url, headers=headers, **kwargs)

    def post(self, url, **kwargs):
        """Make POST request with automatic authentication/refresh"""
        self._ensure_authenticated()
        headers = kwargs.pop('headers', {})
        headers['Authorization'] = f'Bearer {self.access_token}'
        headers['Content-Type'] = headers.get('Content-Type', 'application/json')
        return requests.post(url, headers=headers, **kwargs)

## 3. Probe configuration

**Verified from Swagger (200 OK):** `view-details` takes params
`viewName`, `databaseName`, and `serverId` — e.g.
`/public/api/view-details?databaseName=dataportal&serverId=1&viewName=aa1`.

**Note:** the Swagger test that returned 200 was against **prod**
(`datacatalog.lanl.gov`); the dev listing response uses `db` instead of
`databaseName`, so dev/prod schemas differ slightly. The probe targets dev —
if the smoke test still 401s on dev with the corrected params, that's a
dev-specific auth issue to raise with Maxen.

In [3]:

# CONFIG

BASE_URL = "https://datacatalog-d.lanl.gov/denodo-data-catalog"  # dev
TARGET_DB = "dataportal"

LIST_VIEWS_PATH = "/public/api/views"          # verified: params serverId=1
VIEW_DETAILS_PATH = "/public/api/view-details" # verified: viewName, databaseName, serverId
SERVER_ID = 1

SLEEP_BETWEEN_CALLS = 0.15  # seconds; ~6-7 req/s max, sequential
CHECKPOINT_EVERY = 500      # commit + progress print interval
DB_PATH = "probe_results.db"

# Property names that suggest files/docs even without a URL in the value.
RESOURCE_NAME_KEYWORDS = [
    "attachment", "file", "document", "doc", "download",
    "resource", "link", "url", "report", "manual", "readme",
]

# URL classification: people-directory and contact links are not
# layer-3 *resources* (documents/files) — count them separately so the
# Maxen number isn't inflated.
PERSON_LINK_MARKERS = ("pbplus.lanl.gov",)

URL_RE = re.compile(r'https?://[^\s"\'<>]+')
NAME_KEYWORD_RE = re.compile(
    "|".join(re.escape(k) for k in RESOURCE_NAME_KEYWORDS), re.IGNORECASE
)

## 4. Authenticate (browser popup — once)

`webview.start()` blocks until the popup closes. This is the only interactive step;
refresh during the long sweep is silent and automatic.

In [4]:
auth_manager = OAuthManager(
    client_id=os.getenv('AUTH_FLOW_CLIENT_ID'),
    client_secret=os.getenv('AUTH_FLOW_CLIENT_SECRET'),
    redirect_uri=os.getenv('REDIRECT_URI'),
    auth_url=os.getenv('AUTH_URL'),
    token_url=os.getenv('TOKEN_URL'),
    scope=os.getenv('SCOPE'),
)

auth_manager.authenticate()

Authenticating...
✓ Captured Authorization
✓ Captured Authorization
✓ Authentication successful


True

## 5. Denodo client

Both endpoints Swagger-verified. `view-details` takes `viewName` + `databaseName` +
`serverId` (NOT `name`/`db` — an earlier guess that produced 401s). `list_views`
fetches once with `serverId` and filters client-side on the `db` field (dev's
AllElementsDto uses `db`, not `databaseName`).

In [9]:
class DenodoClient:
    def __init__(self, auth_manager):
        self.auth = auth_manager

    def _get(self, path, params=None, retries=3):
        url = BASE_URL.rstrip("/") + path
        for attempt in range(1, retries + 1):
            try:
                resp = self.auth.get(url, params=params, timeout=30)
                resp.raise_for_status()
                return resp.json()
            except (requests.RequestException, ValueError) as exc:
                if attempt == retries:
                    raise
                wait = 2 ** attempt
                print(f"  [warn] {exc} \u2014 retry {attempt}/{retries} in {wait}s")
                time.sleep(wait)

    def list_views(self, db):
        """Fetch all views via serverId, filter to `db` client-side.

        Dev response items: {id, name, value, description, db,
        elementType, elementSubtype, path, lastModificationDate,
        fields, deleted}. NOTE: dev uses 'db', prod uses 'databaseName'
        in this response — handle both.
        """
        data = self._get(LIST_VIEWS_PATH, params={"serverId": SERVER_ID})
        for item in data:
            item_db = item.get("db") or item.get("databaseName")
            if item_db == db and not item.get("deleted"):
                name = item.get("name")
                if name:
                    yield name

    def get_view_details(self, name, db):
        # Verified param names from Swagger 200:
        return self._get(
            VIEW_DETAILS_PATH,
            params={"viewName": name, "databaseName": db, "serverId": SERVER_ID},
        )

## 6. Signal extraction

Rewritten for the **real** `view-details` response shape (verified via Swagger):
properties live under `propertyInfo.summaryPropertyMap` / `generalTabPropertyMap` /
`customTabPropertyMap` as `{groupName: [property, ...]}`, with the value in
`visualValue` (HTML for RICH_TEXT). The top-level `description` can also be RICH_TEXT.

Two important details:
- **URLs are extracted from the RAW value before HTML stripping** — links live in
  `href="..."` attributes, which stripping would destroy.
- **People-directory links** (pbplus.lanl.gov) are classified separately as
  `person_link` — they're contacts, not layer-3 document resources, and shouldn't
  inflate the headline number.
- `connectionUris` (JDBC/ODBC/REST boilerplate) is deliberately ignored.

In [10]:
def strip_html(text):
    """Remove tags and unescape entities (RICH_TEXT values embed HTML)."""
    return html.unescape(re.sub(r"<[^>]+>", " ", text or ""))


def classify_url(url):
    """Separate people-directory/contact links from real resource URLs."""
    if any(marker in url for marker in PERSON_LINK_MARKERS):
        return "person_link"
    return "resource_url"


def iter_properties(details):
    """Yield (group_name, property_dict) across all three property maps."""
    prop_info = details.get("propertyInfo") or {}
    for map_key in ("summaryPropertyMap", "generalTabPropertyMap",
                    "customTabPropertyMap"):
        for group_name, props in (prop_info.get(map_key) or {}).items():
            for prop in props or []:
                yield group_name, prop


def extract_signals(details):
    """Return list of (signal_type, property_name, evidence) tuples."""
    signals = []

    # Properties (propertyInfo maps) 
    for group_name, prop in iter_properties(details):
        pname = str(prop.get("propertyName", ""))
        raw = str(prop.get("visualValue") or "")

        # URLs: extract from RAW html (hrefs die if we strip first),
        # then unescape entities like &#64;
        for url in URL_RE.findall(html.unescape(raw)):
            signals.append((classify_url(url), f"{group_name}/{pname}", url))

        # Property name suggests files/docs (even without a URL)
        cleaned = strip_html(raw).strip()
        if NAME_KEYWORD_RE.search(pname) and cleaned:
            signals.append(("resource_named_property",
                            f"{group_name}/{pname}", cleaned[:300]))

    # Top-level description (can be RICH_TEXT)
    desc_raw = str(details.get("description") or "")
    for url in URL_RE.findall(html.unescape(desc_raw)):
        signals.append((classify_url(url), "description", url))

    # NOTE: connectionUris deliberately ignored (JDBC/ODBC boilerplate,
    # not layer-3 resources).
    return signals

## 7. SQLite setup — results + checkpoint

Two tables: `hits` (raw evidence rows) and `processed` (checkpoint, primary-keyed on
`(view_name, db)` so reruns skip completed views).

In [11]:
def init_db(path):
    conn = sqlite3.connect(path)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS hits (
            view_name     TEXT,
            db            TEXT,
            signal_type   TEXT,
            property_name TEXT,
            evidence      TEXT
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS processed (
            view_name TEXT,
            db        TEXT,
            n_signals INTEGER,
            PRIMARY KEY (view_name, db)
        )
    """)
    conn.commit()
    return conn

## 8. Smoke test — probe 5 views first

Run this **before** the full loop to confirm the token works against the Data Catalog
API (your earlier test hit the Denodo *server* RESTful web service — different API,
usually the same token, but worth verifying) and that response shapes match. If
`list_views` returns nothing or `view-details` errors, fix the endpoint paths /
param names / response keys above first.

In [8]:
client = DenodoClient(auth_manager)

# Grab just the first few views to verify shapes
sample_views = []
for v in client.list_views(TARGET_DB):
    if v:
        sample_views.append(v)
    if len(sample_views) >= 5:
        break

print("Sample views:", sample_views)

if sample_views:
    details = client.get_view_details(sample_views[0], TARGET_DB)
    print("\nTop-level keys in view-details response:", list(details.keys()))
    print("\nSignals found:", extract_signals(details))

Sample views: ['aa1', 'aa1_locale', 'aa2', 'aa3', 'aa4']

Top-level keys in view-details response: ['id', 'name', 'databaseName', 'schema', 'totalFields', 'inIndex', 'readPermission', 'inLocal', 'inVDP', 'tags', 'categories', 'endorsements', 'warnings', 'deprecations', 'description', 'descriptionType', 'showStatistics', 'statistics', 'association', 'searchForm', 'field', 'propertyInfo', 'viewStatisticsInfo', 'savedQuery', 'connectionUris', 'hasRequests']

Signals found: [('person_link', 'Details/Primary Point of Contact', 'https://pbplus.lanl.gov/search/basia@lanl.gov'), ('person_link', 'Details/Data Owner', 'https://pbplus.lanl.gov/search/x182295_20250707@win.lanl.gov')]


## 9. Main probe loop (resumable) - Set limit=50 for a dry run; limit=None for the full sweep

Safe to interrupt (Kernel \u2192 Interrupt) and rerun — it resumes from the checkpoint
table., ~15k calls take roughly **5 hours**; the OAuthManager
refreshes the token automatically when it expires mid-run.

In [12]:
def run_probe(limit=None):
    """Set limit=50 for a dry run; limit=None for the full sweep."""
    client = DenodoClient(auth_manager)
    conn = init_db(DB_PATH)

    already_done = {
        (row[0], row[1])
        for row in conn.execute("SELECT view_name, db FROM processed")
    }
    print(f"Resuming: {len(already_done)} views already processed.")

    print(f"Listing views in '{TARGET_DB}' ...")
    all_views = [v for v in client.list_views(TARGET_DB) if v]
    print(f"Found {len(all_views)} views total.")

    todo = [v for v in all_views if (v, TARGET_DB) not in already_done]
    if limit:
        todo = todo[:limit]
    print(f"{len(todo)} views to probe this run.\n")

    signal_counter = Counter()
    views_with_resources = 0
    start = time.time()

    for i, view_name in enumerate(todo, 1):
        try:
            details = client.get_view_details(view_name, TARGET_DB)
            signals = extract_signals(details)
        except Exception as exc:
            print(f"  [error] {view_name}: {exc} \u2014 skipping")
            signals = []

        for sig_type, pname, evidence in signals:
            conn.execute(
                "INSERT INTO hits VALUES (?, ?, ?, ?, ?)",
                (view_name, TARGET_DB, sig_type, pname, evidence),
            )
            signal_counter[sig_type] += 1

        if signals:
            views_with_resources += 1
        conn.execute(
            "INSERT OR REPLACE INTO processed VALUES (?, ?, ?)",
            (view_name, TARGET_DB, len(signals)),
        )

        if i % CHECKPOINT_EVERY == 0:
            conn.commit()
            rate = i / (time.time() - start)
            eta_min = (len(todo) - i) / rate / 60 if rate else 0
            print(
                f"[{i}/{len(todo)}] "
                f"{views_with_resources} resource-bearing so far | "
                f"{rate:.1f} views/s | ETA ~{eta_min:.0f} min"
            )

        time.sleep(SLEEP_BETWEEN_CALLS)

    conn.commit()
    conn.close()
    print(f"\nDone. {views_with_resources} resource-bearing views this run.")
    print("Signal breakdown (this run):")
    for sig_type, count in signal_counter.most_common():
        print(f"  {sig_type:28s} {count}")

In [13]:
# Dry run first:
run_probe(limit=None)

# Then the full sweep (comment the line above, uncomment below):
# run_probe()

Resuming: 3550 views already processed.
Listing views in 'dataportal' ...
Found 15007 views total.
11457 views to probe this run.

[500/11457] 5 resource-bearing so far | 0.7 views/s | ETA ~245 min
[1000/11457] 6 resource-bearing so far | 0.7 views/s | ETA ~234 min
[1500/11457] 6 resource-bearing so far | 0.7 views/s | ETA ~222 min
[2000/11457] 26 resource-bearing so far | 0.7 views/s | ETA ~211 min
[2500/11457] 34 resource-bearing so far | 0.7 views/s | ETA ~203 min
[3000/11457] 35 resource-bearing so far | 0.7 views/s | ETA ~195 min
[3500/11457] 179 resource-bearing so far | 0.7 views/s | ETA ~184 min
[4000/11457] 616 resource-bearing so far | 0.7 views/s | ETA ~172 min
[4500/11457] 1086 resource-bearing so far | 0.7 views/s | ETA ~160 min
[5000/11457] 1359 resource-bearing so far | 0.7 views/s | ETA ~151 min
✓ Token refreshed successfully
[5500/11457] 1381 resource-bearing so far | 0.7 views/s | ETA ~140 min
[6000/11457] 1599 resource-bearing so far | 0.7 views/s | ETA ~130 min
[650

##  splits it: views with any signal vs. views with true resource signals (resource_url / resource_named_property), and prints the actual 4 resource URLs so you can eyeball whether they're genuine documents (SharePoint, wiki, PDFs) — gold-standard evidence for the deck if so. If everything looks right, flip to run_probe() and let the full sweep run ~5 hours

In [12]:
conn = sqlite3.connect(DB_PATH)

total = conn.execute("SELECT COUNT(*) FROM processed").fetchone()[0]

# Views with ANY signal (including person_link)
any_signal = conn.execute(
    "SELECT COUNT(*) FROM processed WHERE n_signals > 0").fetchone()[0]

# Views with TRUE resource signals (excluding person_link)
true_resources = conn.execute("""
    SELECT COUNT(DISTINCT view_name) FROM hits
    WHERE signal_type IN ('resource_url', 'resource_named_property')
""").fetchone()[0]

print(f"Of {total} views probed:")
print(f"  {any_signal} have any layer-3 signal (incl. contact links)")
print(f"  {true_resources} have TRUE resource signals (docs/files/URLs)")

# What are those 4 resource URLs? Worth eyeballing:
print("\nTrue resource URLs found:")
for row in conn.execute("""
    SELECT view_name, property_name, evidence FROM hits
    WHERE signal_type = 'resource_url' LIMIT 20
"""):
    print(f"  {row[0]} | {row[1]}\n    -> {row[2]}")

conn.close()

Of 50 views probed:
  39 have any layer-3 signal (incl. contact links)
  2 have TRUE resource signals (docs/files/URLs)

True resource URLs found:
  admin_option_type_fvts | Additional Information/More Help for Web Services
    -> https://collaborate.lanl.gov/x/tYV4Cw
  admin_option_type_fvts | Additional Information/More Help for Web Services
    -> https://collaborate.lanl.gov/x/tYV4Cw
  affiliation_type_fvts | Details/Access Role Request
    -> https://accessit.lanl.gov
  affiliation_type_fvts | Details/Access Role Request
    -> https://accessit.lanl.gov


## 10. Summary — After running all of the views, the results are as follow: 

Rerun this cell any time; it reads from `probe_results.db` across all runs.

In [14]:
conn = sqlite3.connect(DB_PATH)

total_processed = conn.execute("SELECT COUNT(*) FROM processed").fetchone()[0]
total_with = conn.execute(
    "SELECT COUNT(*) FROM processed WHERE n_signals > 0"
).fetchone()[0]

print("=" * 60)
print(f"Of the {total_processed} views probed on dev, "
      f"{total_with} have layer-3 resource signals.")

print("\nBreakdown by signal type (all runs):")
for sig_type, count in conn.execute("""
    SELECT signal_type, COUNT(*) FROM hits
    GROUP BY signal_type ORDER BY COUNT(*) DESC
"""):
    print(f"  {sig_type:28s} {count}")

print("\nTop properties carrying signals:")
for pname, count in conn.execute("""
    SELECT property_name, COUNT(*) FROM hits
    GROUP BY property_name ORDER BY COUNT(*) DESC LIMIT 15
"""):
    print(f"  {pname:40s} {count}")

conn.close()

Of the 15007 views probed on dev, 5366 have layer-3 resource signals.

Breakdown by signal type (all runs):
  person_link                  9436
  resource_named_property      1348
  resource_url                 1117

Top properties carrying signals:
  Details/Primary Point of Contact         5084
  Details/Data Owner                       4351
  ODS Details/ODS DB Link                  1347
  Details/Access Role Request              558
  description                              544
  AI Portal/A.I. Summary                   9
  Details/Access Request                   3
  Additional Information/More Help for Web Services 2
  API Information/Base URL Path:           2
  API Information/OpenAPI Reference:       1


## 11. Example hits (for the meeting deck)

Pull a few concrete `(view, property, url)` rows to show as evidence.

In [15]:
conn = sqlite3.connect(DB_PATH)

print("Sample URL hits:\n")
for row in conn.execute("""
    SELECT view_name, property_name, evidence FROM hits
    WHERE signal_type = 'url_in_value' LIMIT 10
"""):
    print(f"  view={row[0]}\n    property={row[1]}\n    url={row[2]}\n")

conn.close()

Sample URL hits:



In [16]:
conn = sqlite3.connect(DB_PATH)

total = conn.execute("SELECT COUNT(*) FROM processed").fetchone()[0]
any_sig = conn.execute(
    "SELECT COUNT(*) FROM processed WHERE n_signals > 0").fetchone()[0]
true_res = conn.execute("""
    SELECT COUNT(DISTINCT view_name) FROM hits
    WHERE signal_type IN ('resource_url', 'resource_named_property')
""").fetchone()[0]
url_only = conn.execute("""
    SELECT COUNT(DISTINCT view_name) FROM hits
    WHERE signal_type = 'resource_url'
""").fetchone()[0]

print(f"Of {total} views probed on dev:")
print(f"  {any_sig} have any layer-3 signal (incl. contact links)")
print(f"  {true_res} have true resource signals")
print(f"  {url_only} have actual resource URLs")

print("\nTop properties carrying true resources:")
for row in conn.execute("""
    SELECT property_name, COUNT(*) FROM hits
    WHERE signal_type = 'resource_url'
    GROUP BY property_name ORDER BY COUNT(*) DESC LIMIT 10
"""):
    print(f"  {row[0]:50s} {row[1]}")
conn.close()

Of 15007 views probed on dev:
  5366 have any layer-3 signal (incl. contact links)
  2024 have true resource signals
  829 have actual resource URLs

Top properties carrying true resources:
  Details/Access Role Request                        558
  description                                        544
  AI Portal/A.I. Summary                             9
  Details/Access Request                             2
  Additional Information/More Help for Web Services  2
  API Information/OpenAPI Reference:                 1
  API Information/Base URL Path:                     1


## Check if  ODS DB Link is just a database?

In [17]:
import sqlite3
from collections import Counter

conn = sqlite3.connect(DB_PATH)

# Look at the actual values stored for this property
rows = conn.execute("""
    SELECT view_name, evidence FROM hits
    WHERE property_name = 'ODS Details/ODS DB Link'
    LIMIT 30
""").fetchall()

for view, evidence in rows[:15]:
    print(f"  {view:35s} -> {evidence[:80]}")

# Key test: does ANY value contain an actual URL?
url_count = conn.execute("""
    SELECT COUNT(*) FROM hits
    WHERE property_name = 'ODS Details/ODS DB Link'
      AND evidence LIKE '%http%'
""").fetchone()[0]
print(f"\nValues containing http URLs: {url_count} / 1347")

# How many distinct values? Few distinct values = it's an enum-like
# field (database names), not per-view document links
distinct = conn.execute("""
    SELECT evidence, COUNT(*) FROM hits
    WHERE property_name = 'ODS Details/ODS DB Link'
    GROUP BY evidence ORDER BY COUNT(*) DESC LIMIT 10
""").fetchall()
print("\nTop 10 most common values:")
for val, n in distinct:
    print(f"  {n:5d}x  {val[:70]}")

conn.close()

  announcement                        -> FVTS_LINK
  ap_accounting_events_all            -> EBS_LINK
  ap_ae_headers_all                   -> EBS_LINK
  ap_aging_period_lines               -> EBS_LINK
  ap_aging_periods                    -> EBS_LINK
  ap_allocation_rule_lines            -> EBS_LINK
  ap_allocation_rules                 -> EBS_LINK
  ap_aud_auditors                     -> EBS_LINK
  ap_awt_group_taxes_all              -> EBS_LINK
  ap_awt_groups                       -> EBS_LINK
  ap_awt_tax_rates_all                -> EBS_LINK
  ap_bank_account_uses_all            -> EBS_LINK
  ap_bank_accounts_all                -> EBS_LINK
  ap_bank_branches                    -> EBS_LINK
  ap_batches_all                      -> EBS_LINK

Values containing http URLs: 0 / 1347

Top 10 most common values:
    852x  EBS_LINK
    182x  NOLINK
    137x  SNFLWR_LINK
     99x  ARIBA_LINK
     35x  AFM_LINK
      8x  LMS_LINK
      8x  Footprints
      8x  FVTS_LINK
      5x  INDYSOFT_LINK


### Conclusion 1: ODS Details/ODS DB Link was flagged by keyword matching ("Link") but verified to contain Oracle database-link identifiers (EBS_LINK 63%, SNFLWR_LINK, ARIBA_LINK, etc.; 0 of 1,347 values contain a URL; 182 are literally "NOLINK"). This property is source-system lineage metadata and belongs to Layer 1/2 provenance, not Layer 3 resources. The verified Layer-3 population is the 829 views (5.5%) carrying embedded URLs.

## Check 2 — Are the URLs in Access Role Request actual documents, or just access-request forms?

In [18]:
import sqlite3
from urllib.parse import urlparse
from collections import Counter

conn = sqlite3.connect(DB_PATH)

# Sample 20 random URLs (random, not LIMIT — avoids alphabetical bias
# where the first 20 all come from the same view family)
rows = conn.execute("""
    SELECT view_name, evidence FROM hits
    WHERE property_name = 'Details/Access Role Request'
      AND signal_type = 'resource_url'
    ORDER BY RANDOM() LIMIT 20
""").fetchall()

print("Sample of 20 URLs:\n")
for view, url in rows:
    print(f"  {view:35s} {url[:90]}")

# Aggregate: which domains do ALL 558 URLs point to?
all_urls = conn.execute("""
    SELECT evidence FROM hits
    WHERE property_name = 'Details/Access Role Request'
      AND signal_type = 'resource_url'
""").fetchall()

domains = Counter(urlparse(u[0]).netloc for u in all_urls)
print("\nDomain distribution (all 558 URLs):")
for domain, n in domains.most_common(15):
    print(f"  {n:5d}x  {domain}")

conn.close()

Sample of 20 URLs:

  wtype_signatordef                   https://accessit.lanl.gov
  vlanlrailtasktrackerreviewerreassignmentcommentsbase https://accessit.lanl.gov
  shipping_cat_reference              https://weather.lanl.gov
  ta4                                 https://accessit.lanl.gov
  ww_forcemain                        https://accessit.lanl.gov
  units_time                          https://accessit.lanl.gov
  time_dim                            https://accessit.lanl.gov
  tvl_orgs                            https://accessit.lanl.gov
  state                               https://accessit.lanl.gov
  pa_report_types                     https://accessit.lanl.gov
  passport_tideceqn                   https://accessit.lanl.gov
  trafficcameras                      https://accessit.lanl.gov
  zd_units_volume                     https://accessit.lanl.gov
  zd_u_mob_comment                    https://accessit.lanl.gov
  wtype_comment                       https://accessit.lanl.gov
  ta

### Conclusion 2: Details/Access Role Request URLs were verified: 93% (520/558) point to accessit.lanl.gov, LANL's access-request portal, with near-identical values repeated across hundreds of views. These are access instructions, not document resources — excluded from Layer 3 and mapped instead to a dedicated access_instructions field (which itself is useful: DSI can tell users exactly how to request access to each source). The sole remaining Layer-3 resource carrier is the view 'description' field (544 embedded URLs), pending its own domain verification.

## Check 3 - Check 'description'

In [20]:
conn = sqlite3.connect(DB_PATH)

rows = conn.execute("""
    SELECT view_name, evidence FROM hits
    WHERE property_name = 'description'
      AND signal_type = 'resource_url'
    ORDER BY RANDOM() LIMIT 20
""").fetchall()
print("Sample of 20 description URLs:\n")
for view, url in rows:
    print(f"  {view:35s} {url[:90]}")

all_urls = conn.execute("""
    SELECT evidence FROM hits
    WHERE property_name = 'description'
      AND signal_type = 'resource_url'
""").fetchall()
from urllib.parse import urlparse
from collections import Counter
domains = Counter(urlparse(u[0]).netloc for u in all_urls)
print("\nDomain distribution (all 544 URLs):")
for domain, n in domains.most_common(15):
    print(f"  {n:5d}x  {domain}")

# And the final headline number: distinct views with description URLs
n_views = conn.execute("""
    SELECT COUNT(DISTINCT view_name) FROM hits
    WHERE property_name = 'description' AND signal_type = 'resource_url'
""").fetchone()[0]
print(f"\nDistinct views with URLs in description: {n_views}")
conn.close()

Sample of 20 description URLs:

  ap_pol_violations_gt                https://etrm.live/etrm-12.1.1/etrm.oracle.com/pls/et1211d9/etrm_pnavfc44.html
  ap_expense_feed_lines_all           https://etrm.live/etrm-12.2.2/etrm.oracle.com/pls/trm1222/etrm_pnav647e.html?c_name=AP_EXP
  bv_ap_cards_all                     https://etrm.live/etrm-12.2.2/etrm.oracle.com/pls/trm1222/etrm_pnav2c2b.html?c_name=AP_CAR
  ta16b_24                            https://weather.lanl.gov
  bv_ap_duplicate_vendors_all         https://etrm.live/etrm-12.2.2/etrm.oracle.com/pls/trm1222/etrm_pnav5f98.html?c_name=AP_DUP
  bv_ap_card_codes_all                https://etrm.live/etrm-12.2.2/etrm.oracle.com/pls/trm1222/etrm_pnavf87d.html?c_name=AP_CAR
  ta54b_15                            https://weather.lanl.gov
  ap_aging_periods                    https://etrm.live/etrm-12.2.2/etrm.oracle.com/pls/trm1222/etrm_pnav9a18.html?c_name=AP_AGI
  per_addresses                       https://docs.oracle.com/en/cloud/saas/human

### Conclusion 3: 

1. Description URLs are genuine technical documentation. 86% (470 (399+71) of 544) point to Oracle table-level documentation — etrm.live (399) and docs.oracle.com (71). The pattern is precise: each view links to the data-dictionary page of its exact source table (e.g., view ap_aging_periods → the eTRM page for Oracle E-Business Suite table AP_AGING_PERIODS; per_addresses → the Oracle Cloud HCM dictionary for that same table). This is per-table data-dictionary documentation — exactly the kind of "additional files" the tri-layer model's Layer 3 envisions. It also corroborates the first verification: the views carrying EBS_LINK as their source system are the same views carrying eTRM documentation.
2. A clean 1:1 relationship. 544 URLs across 544 distinct views — exactly one documentation link per view, never more. This simplifies the schema: Layer 3 can be a single nullable documentation_url field rather than a resources[] array.
3. Two data-quality caveats. First, etrm.live is a third-party mirror (Oracle retired the official etrm.oracle.com), so 73% of documentation links depend on an unofficial site — they work today, but with no availability guarantee; three more links point to a personal blog (oracleapps88.blogspot.com). Second, two URLs point to secure.coomeva.com.co, an unrelated Colombian company domain — almost certainly data-entry errors, worth flagging to data stewards.